'''
Source: Impact4cast (Max Planck Institute)
Original code from the Max Planck Institute for Informatics / Max Planck Institute for Security and Privacy research group.

Modifications: Bug fixes for checkpoint resume mechanism and dataset adaptation for scientific entity impact prediction.
'''


cell1不用跑，cell2和3需要跑

In [ ]:
import glob
import gzip
import json
import os
import time
import pickle
import random
import re
from datetime import datetime, date
from functools import reduce
import ahocorasick  # pip install pyahocorasick
import json

# -----------------------------
# 函数定义
# -----------------------------
def get_single_article_string(article):
    # 安全地获取标题和摘要
    curr_title = article.get('title') or ''
    abstract_inverted_index = article.get('abstract_inverted_index') or {}

    # 展平 inverted index
    position_word_list = [
        (position, word)
        for word, positions in abstract_inverted_index.items()
        for position in positions
    ]

    # 组装摘要文本
    if position_word_list:
        sorted_abstract = sorted(position_word_list)
        curr_abstract = ' '.join(word for position, word in sorted_abstract)
    else:
        curr_abstract = ''

    # 清理特殊符号
    replace_pairs = [
        ['\n', ' '], ['-', ' '], [' \" a', 'oa'], ['\" a', 'ae'], ['\"a', 'ae'],
        [' \" o', 'oe'], ['\" o', 'oe'], ['\"o', 'oe'], [' \" u', 'ue'], ['\" u', 'ue'],
        ['\"u', 'ue'], [' \' a', 'a'], [' \' e', 'e'], [' \' o', 'o'],
        ["\' ", ""], ["\'", ""], ['  ', ' '], ['  ', ' ']
    ]

    # 拼接标题 + 摘要
    article_string = (curr_title + ' ' + curr_abstract).lower()
    article_string = reduce(lambda text, pair: text.replace(pair[0], pair[1]), replace_pairs, article_string)

    return article_string



def get_date_and_part_from_path(path):
    date_folder = os.path.dirname(path)
    date_str = date_folder.split('=')[-1]
    file_name = os.path.basename(path)
    part_str = file_name.split('_')[-1].split('.')[0]
    return date_str, int(part_str)


def extract_id(filename):
    match = re.search(r'log_edge_part_(\d+)_', filename)
    if match:
        return int(match.group(1))
    return None


# -----------------------------
# 路径定义
# -----------------------------
log_folder = 'logs_pair'
edge_folder = "H:\\concept_pair"
edge_folder_log = 'concept_pair_log'
data_folder = "data_concept_graph"
cwd = os.getcwd()
parent_dir = os.path.dirname(cwd)
concept_folder = os.path.join(parent_dir, data_folder)
progress_file = "progress.json"

for folder in [log_folder, edge_folder, edge_folder_log]:
    os.makedirs(folder, exist_ok=True)

# -----------------------------
# 文件路径筛选
# -----------------------------
base_folder = "G:\\openalex-snapshot\\data\\works"
file_paths = glob.glob(f'{base_folder}/updated_date=*/part_*.gz')
file_paths = sorted(file_paths, key=get_date_and_part_from_path)

start_date = datetime.strptime("2022-12-20", "%Y-%m-%d")
end_date = datetime.strptime("2025-10-01", "%Y-%m-%d")

curr_run_file_paths = [
    path for path in file_paths
    if start_date <= datetime.strptime(get_date_and_part_from_path(path)[0], "%Y-%m-%d") <= end_date
]

# -----------------------------
# 加载概念词并构建 Aho–Corasick 自动机
# -----------------------------
concept_folder = "H:\\concept_folder"
concepts_files = os.path.join(concept_folder, 'final_concepts.txt')
with open(concepts_files, 'r', encoding='utf-8') as file:
    full_concepts = [concept.strip().lower() for concept in file.readlines() if concept.strip()]

A = ahocorasick.Automaton()
for i, concept in enumerate(full_concepts):
    A.add_word(concept, i)
A.make_automaton()

print(f"✅ 加载 {len(full_concepts)} 个概念，自动机构建完成。")

# -----------------------------
# 断点续跑设置
# -----------------------------
if os.path.exists(progress_file):
    with open(progress_file, "r") as f:
        finished_ids = set(json.load(f))
else:
    finished_ids = set()

paper_starting_date = date(1990, 1, 1)
batch_size = 5  # 每批处理多少文件
batch_save_every = 50000  # 每隔多少条写入一次 gzip 文件

remaining_ids = [i for i in range(len(curr_run_file_paths)) if i not in finished_ids]

# -----------------------------
# 主循环
# -----------------------------
for curr_ID in remaining_ids[:batch_size]:
    formatted_ID = '{:03d}'.format(curr_ID)
    file_path = curr_run_file_paths[curr_ID]

    edge_file = os.path.join(edge_folder, f'edge_part_{formatted_ID}.gz')
    edge_file_log = os.path.join(edge_folder_log, f'edge_part_{formatted_ID}.txt')
    log_file_txt = os.path.join(log_folder, f'log_edge_part_{formatted_ID}.txt')
    log_file_txt_finish = os.path.join(log_folder, f'log_edge_part_{formatted_ID}_finish.txt')

    current_time = datetime.now()
    with open(log_file_txt, 'a') as log_file:
        log_file.write(f'Current time: {current_time}; Start File: {file_path}\n')

    edge_lists = []
    with gzip.open(file_path, 'rt', encoding='utf-8') as file:
        lines = file.readlines()

        if not lines:
            with open(log_file_txt, 'a') as log_file:
                log_file.write("⚠️ File empty\n")
            continue

        for id_line, line in enumerate(lines):
            article_object = json.loads(line)
            get_date = datetime.strptime(article_object['publication_date'], "%Y-%m-%d").date()
            curr_paper_time = (get_date - paper_starting_date).days
            curr_all_citations = article_object.get('cited_by_count', 0)
            curr_citations_per_year = article_object.get('counts_by_year', [])
            curr_article = get_single_article_string(article_object)

            # 使用 Aho–Corasick 快速匹配概念
            concepts_for_single_paper = [idx for _, idx in A.iter(curr_article)]

            for ii in range(len(concepts_for_single_paper)):
                for jj in range(ii + 1, len(concepts_for_single_paper)):
                    edge_lists.append([
                        concepts_for_single_paper[ii],
                        concepts_for_single_paper[jj],
                        curr_paper_time,
                        curr_all_citations,
                        curr_citations_per_year
                    ])

            if len(edge_lists) >= batch_save_every:
                with gzip.open(edge_file, 'ab') as output_file:
                    pickle.dump(edge_lists, output_file)
                edge_lists = []  # 清空缓存

            if id_line % 5000 == 0:
                with open(log_file_txt, 'a') as log_file:
                    log_file.write(f"Processed {id_line}/{len(lines)} lines\n")

    # 保存剩余数据
    if edge_lists:
        with gzip.open(edge_file, 'ab') as output_file:
            pickle.dump(edge_lists, output_file)

    with open(edge_file_log, 'a') as log_file:
        log_file.write(f'\nedge_list={len(edge_lists)}')

    with open(log_file_txt_finish, 'a') as log_file:
        log_file.write(f'Finish File: {file_path}; Time: {datetime.now()}\n')

    # 更新进度
    finished_ids.add(curr_ID)
    with open(progress_file, "w") as f:
        json.dump(list(finished_ids), f)

print(f"🎉 所有批次完成（共 {len(finished_ids)} 个文件）")


### 创建edge_part_XXX.gz

In [ ]:
import glob
import gzip
import json
import os
import pickle
import re
from datetime import datetime, date
from functools import reduce

# =====================================================
# AC AUTOMATON
# =====================================================
class ACAutomaton:
    def __init__(self):
        self.goto = [{}]
        self.out = [set()]
        self.fail = [0]

    def add_word(self, word, index):
        state = 0
        for char in word:
            if char not in self.goto[state]:
                self.goto[state][char] = len(self.goto)
                self.goto.append({})
                self.out.append(set())
                self.fail.append(0)
            state = self.goto[state][char]
        self.out[state].add(index)

    def build(self):
        queue = []
        for char, state in self.goto[0].items():
            queue.append(state)
        while queue:
            r = queue.pop(0)
            for char, s in self.goto[r].items():
                queue.append(s)
                f = self.fail[r]
                while f and char not in self.goto[f]:
                    f = self.fail[f]
                self.fail[s] = self.goto[f].get(char, 0)
                self.out[s] |= self.out[self.fail[s]]

    def search(self, text):
        state = 0
        results = set()
        for char in text:
            while state and char not in self.goto[state]:
                state = self.fail[state]
            state = self.goto[state].get(char, 0)
            if self.out[state]:
                results |= self.out[state]
        return results

# =====================================================
# HELPERS
# =====================================================
def safe_json_loads(line):
    try:
        return json.loads(line)
    except:
        return None

def safe_date_parsing(date_str):
    try:
        return datetime.strptime(date_str, "%Y-%m-%d").date()
    except:
        return None

def get_single_article_string(article):
    try:
        title = article.get('title', '') or ''
        inv = article.get('abstract_inverted_index')
        if not inv:
            abstract = ""
        else:
            pairs = []
            for w, ps in inv.items():
                for p in ps or []:
                    pairs.append((p, w))
            abstract = ' '.join(w for _, w in sorted(pairs))

        replace_pairs = [
            ['\n', ' '], ['-', ' '], ['  ', ' ']
        ]

        text = (title + ' ' + abstract).lower()
        for a, b in replace_pairs:
            text = text.replace(a, b)

        return text
    except:
        return ""

def extract_id(filename):
    m = re.search(r'part_(\d+)', filename)
    return int(m.group(1)) if m else None

# =====================================================
# PROCESS SINGLE FILE
# =====================================================
def process_file(file_path, part_id, edge_folder, log_folder, ac_automaton):
    pid = f"{part_id:03d}"
    edge_file = os.path.join(edge_folder, f'edge_part_{pid}.gz')
    finish_log = os.path.join(log_folder, f'log_edge_part_{pid}_finish.txt')

    # 已完成直接跳过
    if os.path.exists(edge_file) or os.path.exists(finish_log):
        print(f"[{pid}] 已存在，跳过")
        return

    edge_lists = []
    processed, errors = 0, 0
    paper_starting_date = date(1990, 1, 1)

    try:
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                article = safe_json_loads(line)
                if not article:
                    errors += 1
                    continue

                pub_date = safe_date_parsing(article.get('publication_date', ''))
                if not pub_date:
                    errors += 1
                    continue

                text = get_single_article_string(article)
                if not text:
                    continue

                matches = ac_automaton.search(text)
                if len(matches) < 2:
                    continue

                curr_time = (pub_date - paper_starting_date).days
                curr_cite = article.get('cited_by_count', 0)
                curr_cite_year = article.get('counts_by_year', [])

                mc = sorted(matches)
                for i in range(len(mc)):
                    for j in range(i + 1, len(mc)):
                        edge_lists.append([mc[i], mc[j], curr_time, curr_cite, curr_cite_year])

                processed += 1

        # 保存 edge
        with gzip.open(edge_file, 'wb') as f:
            pickle.dump(edge_lists, f)

        # 写完成日志
        with open(finish_log, 'w', encoding='utf-8') as f:
            f.write(f"Finish Time: {datetime.now()}\n")

        print(f"[{pid}] 完成 edges={len(edge_lists)}, papers={processed}, errors={errors}")

    except Exception as e:
        print(f"[{pid}] 处理失败: {e}")


# =====================================================
# ENTRY POINT
# =====================================================
if __name__ == "__main__":
    base_folder = "E:\\2025年科研训练\\openalex-snapshot\\data\\works"
    concept_file = "E:\\2025年科研训练\\concept_folder\\final_concepts.txt"

    edge_folder = "E:\\25年科研训练\\concept_pair"
    log_folder = "E:\\2025年科研训练\\logs_pair"
    os.makedirs(edge_folder, exist_ok=True)
    os.makedirs(log_folder, exist_ok=True)

    files = sorted(glob.glob(f"{base_folder}/updated_date=*/part_*.gz"))

    # =====================================================
    # 加载概念 + 构建 AC 自动机
    # =====================================================
    with open(concept_file, 'r', encoding='utf-8') as f:
        full_concepts = [c.strip().lower() for c in f if c.strip()]
    print(f"概念数量: {len(full_concepts)}")

    ac_automaton = ACAutomaton()
    for idx, concept in enumerate(full_concepts):
        ac_automaton.add_word(concept, idx)
    ac_automaton.build()
    print("AC 自动机构建完成")

    # =====================================================
    # 断点续跑
    # =====================================================
    finished_ids = set()
    for f in glob.glob(os.path.join(log_folder, 'log_edge_part_*_finish.txt')):
        pid = extract_id(f)
        if pid is not None:
            finished_ids.add(pid)
    print(f"已完成 part 数: {len(finished_ids)}")

    # =====================================================
    # 顺序处理每个文件
    # =====================================================
    total_tasks = 0
    for i, f in enumerate(files):
        if i in finished_ids:
            continue
        total_tasks += 1
        process_file(f, i, edge_folder, log_folder, ac_automaton)

    print(f"总任务数: {total_tasks}")
    with open("job_finish.txt", 'a', encoding='utf-8') as f:
        f.write(f"All done: {datetime.now()}\n")

    print("=== 全部任务完成 ===")


In [ ]:
import glob
import gzip
import json
import os
import pickle
import re
from datetime import datetime, date
from functools import reduce
from collections import defaultdict
from ahocorasick import Automaton

# =====================================================
# AC AUTOMATON
# =====================================================
class ACAutomaton:
    def __init__(self):
        self.goto = [{}]
        self.out = [set()]
        self.fail = [0]

    def add_word(self, word, index):
        state = 0
        for char in word:
            if char not in self.goto[state]:
                self.goto[state][char] = len(self.goto)
                self.goto.append({})
                self.out.append(set())
                self.fail.append(0)
            state = self.goto[state][char]
        self.out[state].add(index)

    def build(self):
        queue = []
        for char, state in self.goto[0].items():
            queue.append(state)
        while queue:
            r = queue.pop(0)
            for char, s in self.goto[r].items():
                queue.append(s)
                f = self.fail[r]
                while f and char not in self.goto[f]:
                    f = self.fail[f]
                self.fail[s] = self.goto[f].get(char, 0)
                self.out[s] |= self.out[self.fail[s]]

    def search(self, text):
        state = 0
        results = set()
        for char in text:
            while state and char not in self.goto[state]:
                state = self.fail[state]
            state = self.goto[state].get(char, 0)
            if self.out[state]:
                results |= self.out[state]
        return results

# =====================================================
# HELPERS
# =====================================================
def safe_json_loads(line):
    try:
        return json.loads(line)
    except:
        return None

def safe_date_parsing(date_str):
    try:
        return datetime.strptime(date_str, "%Y-%m-%d").date()
    except:
        return None

def get_single_article_string(article):
    try:
        title = article.get('title', '') or ''
        inv = article.get('abstract_inverted_index')
        if not inv:
            abstract = ""
        else:
            pairs = []
            for w, ps in inv.items():
                for p in ps or []:
                    pairs.append((p, w))
            abstract = ' '.join(w for _, w in sorted(pairs))

        replace_pairs = [
            ['\n', ' '], ['-', ' '], ['  ', ' ']
        ]

        text = (title + ' ' + abstract).lower()
        for a, b in replace_pairs:
            text = text.replace(a, b)

        return text
    except:
        return ""

def extract_id(filename):
    m = re.search(r'part_(\d+)', filename)
    return int(m.group(1)) if m else None

# =====================================================
# PROCESS SINGLE FILE
# =====================================================
def process_file(file_path, part_id, edge_folder, log_folder, ac_automaton):
    pid = f"{part_id:03d}"
    edge_file = os.path.join(edge_folder, f'edge_part_{pid}.gz')
    finish_log = os.path.join(log_folder, f'log_edge_part_{pid}_finish.txt')

    # 已完成直接跳过
    if os.path.exists(edge_file) or os.path.exists(finish_log):
        print(f"[{pid}] 已存在，跳过")
        return

    edge_lists = []
    processed, errors = 0, 0
    paper_starting_date = date(1990, 1, 1)

    try:
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                article = safe_json_loads(line)
                if not article:
                    errors += 1
                    continue

                pub_date = safe_date_parsing(article.get('publication_date', ''))
                if not pub_date:
                    errors += 1
                    continue

                text = get_single_article_string(article)
                if not text:
                    continue

                matches = ac_automaton.search(text)
                if len(matches) < 2:
                    continue

                curr_time = (pub_date - paper_starting_date).days
                curr_cite = article.get('cited_by_count', 0)
                curr_cite_year = article.get('counts_by_year', [])

                mc = sorted(matches)
                for i in range(len(mc)):
                    for j in range(i + 1, len(mc)):
                        edge_lists.append([mc[i], mc[j], curr_time, curr_cite, curr_cite_year])

                processed += 1

        # 保存 edge
        with gzip.open(edge_file, 'wb') as f:
            pickle.dump(edge_lists, f)

        # 写完成日志
        with open(finish_log, 'w', encoding='utf-8') as f:
            f.write(f"Finish Time: {datetime.now()}\n")

        print(f"[{pid}] 完成 edges={len(edge_lists)}, papers={processed}, errors={errors}")

    except Exception as e:
        print(f"[{pid}] 处理失败: {e}")


# =====================================================
# ENTRY POINT
# =====================================================
if __name__ == "__main__":
    base_folder = "G:\\openalex-snapshot\\data\\works"
    concept_file = "E:\\study\\research\\ASIST\\entities.txt"

    edge_folder = "H:\\ASIST\\entities_pair"
    log_folder = "H:\\ASIST\\logs_pair"
    os.makedirs(edge_folder, exist_ok=True)
    os.makedirs(log_folder, exist_ok=True)

    files = sorted(glob.glob(f"{base_folder}/updated_date=*/part_*.gz"))

    # =====================================================
    # 加载概念 + 构建 AC 自动机
    # =====================================================
    with open(concept_file, 'r', encoding='utf-8') as f:
        full_concepts = [c.strip().lower() for c in f if c.strip()]
    print(f"概念数量: {len(full_concepts)}")

    ac_automaton = ACAutomaton()
    for idx, concept in enumerate(full_concepts):
        ac_automaton.add_word(concept, idx)
    ac_automaton.build()
    print("AC 自动机构建完成")

    # =====================================================
    # 断点续跑
    # =====================================================
    finished_ids = set()
    for f in glob.glob(os.path.join(log_folder, 'log_edge_part_*_finish.txt')):
        pid = extract_id(f)
        if pid is not None:
            finished_ids.add(pid)
    print(f"已完成 part 数: {len(finished_ids)}")

    # =====================================================
    # 顺序处理每个文件
    # =====================================================
    total_tasks = 0
    for i, f in enumerate(files):
        if i in finished_ids:
            continue
        total_tasks += 1
        process_file(f, i, edge_folder, log_folder, ac_automaton)

    print(f"总任务数: {total_tasks}")
    with open("job_finish.txt", 'a', encoding='utf-8') as f:
        f.write(f"All done: {datetime.now()}\n")

    print("=== 全部任务完成 ===")


In [ ]:
import glob
import gzip
import json
import os
import pickle
import re
from datetime import datetime, date
from functools import reduce
from collections import defaultdict
from ahocorasick import Automaton

# =====================================================
# AC AUTOMATON
# =====================================================
class ACAutomaton:
    def __init__(self):
        self.goto = [{}]
        self.out = [set()]
        self.fail = [0]

    def add_word(self, word, index):
        state = 0
        for char in word:
            if char not in self.goto[state]:
                self.goto[state][char] = len(self.goto)
                self.goto.append({})
                self.out.append(set())
                self.fail.append(0)
            state = self.goto[state][char]
        self.out[state].add(index)

    def build(self):
        queue = []
        for char, state in self.goto[0].items():
            queue.append(state)
        while queue:
            r = queue.pop(0)
            for char, s in self.goto[r].items():
                queue.append(s)
                f = self.fail[r]
                while f and char not in self.goto[f]:
                    f = self.fail[f]
                self.fail[s] = self.goto[f].get(char, 0)
                self.out[s] |= self.out[self.fail[s]]

    def search(self, text):
        state = 0
        results = set()
        for char in text:
            while state and char not in self.goto[state]:
                state = self.fail[state]
            state = self.goto[state].get(char, 0)
            if self.out[state]:
                results |= self.out[state]
        return results

# =====================================================
# HELPERS
# =====================================================
def safe_json_loads(line):
    try:
        return json.loads(line)
    except:
        return None

def safe_date_parsing(date_str):
    try:
        return datetime.strptime(date_str, "%Y-%m-%d").date()
    except:
        return None

def get_single_article_string(article):
    try:
        title = article.get('title', '') or ''
        inv = article.get('abstract_inverted_index')
        if not inv:
            abstract = ""
        else:
            pairs = []
            for w, ps in inv.items():
                for p in ps or []:
                    pairs.append((p, w))
            abstract = ' '.join(w for _, w in sorted(pairs))

        replace_pairs = [
            ['\n', ' '], ['-', ' '], ['  ', ' ']
        ]

        text = (title + ' ' + abstract).lower()
        for a, b in replace_pairs:
            text = text.replace(a, b)

        return text
    except:
        return ""

def extract_id(filename):
    m = re.search(r'part_(\d+)', filename)
    return int(m.group(1)) if m else None

def extract_part_id_from_path(path):
    """
    从文件路径或文件夹名称中提取part编号
    支持格式: edge_part_123.gz, temp_123, edge_part_123.pkl.gz 等
    """
    basename = os.path.basename(path)
    # 匹配各种格式的part编号
    patterns = [
        r'edge_part_(\d+)\.gz$',     # edge_part_123.gz
        r'temp_(\d+)$',              # temp_123
        
    ]
    
    for pattern in patterns:
        match = re.search(pattern, basename)
        if match:
            return int(match.group(1))
    return None

# =====================================================
# PROCESS SINGLE FILE
# =====================================================
def process_file(file_path, part_id, edge_folder, log_folder, ac_automaton):
    pid = f"{part_id:03d}"
    edge_file = os.path.join(edge_folder, f'edge_part_{pid}.gz')
    finish_log = os.path.join(log_folder, f'log_edge_part_{pid}_finish.txt')
    temp_folder = os.path.join(edge_folder, f'temp_{pid}')
    
    # Checkpoint logic: resume if this part is completed
    if os.path.exists(finish_log) or os.path.exists(edge_file) or os.path.exists(temp_folder):
        print(f"[{pid}] 已存在（edge文件、完成日志或temp文件夹），跳过")
        return
    
    # 创建临时文件夹
    os.makedirs(temp_folder, exist_ok=True)
    
    processed, errors = 0, 0
    paper_starting_date = date(1990, 1, 1)
    
    # 分批处理参数
    batch_size = 5000
    batch_count = 0
    temp_files = []
    current_batch = []
    
    try:
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                article = safe_json_loads(line)
                if not article:
                    errors += 1
                    continue
                
                pub_date = safe_date_parsing(article.get('publication_date', ''))
                if not pub_date:
                    errors += 1
                    continue
                
                text = get_single_article_string(article)
                if not text:
                    continue
                
                matches = ac_automaton.search(text)
                if len(matches) < 2:
                    continue
                
                curr_time = (pub_date - paper_starting_date).days
                curr_cite = article.get('cited_by_count', 0)
                curr_cite_year = article.get('counts_by_year', [])
                
                mc = sorted(matches)
                for i in range(len(mc)):
                    for j in range(i + 1, len(mc)):
                        current_batch.append([mc[i], mc[j], curr_time, curr_cite, curr_cite_year])
                
                processed += 1
                
                # 批次大小达到阈值，保存到临时文件
                if len(current_batch) >= batch_size:
                    temp_file = os.path.join(temp_folder, f'batch_{batch_count:04d}.pkl.gz')
                    with gzip.open(temp_file, 'wb') as tf:
                        pickle.dump(current_batch, tf)
                    temp_files.append(temp_file)
                    current_batch = []
                    batch_count += 1
                    print(f"[{pid}] 已保存批次 {batch_count}，累计处理文章 {processed}")

        # 保存最后一批
        if current_batch:
            temp_file = os.path.join(temp_folder, f'batch_{batch_count:04d}.pkl.gz')
            with gzip.open(temp_file, 'wb') as tf:
                pickle.dump(current_batch, tf)
            temp_files.append(temp_file)
        
        # 写完成日志
        with open(finish_log, 'w', encoding='utf-8') as f:
            f.write(f"Finish Time: {datetime.now()}\n")
            f.write(f"Total batches: {batch_count + 1}\n")
            f.write(f"Processed papers: {processed}\n")
            f.write(f"Errors: {errors}\n")
            f.write(f"Temp folder: {temp_folder}\n")
        
        print(f"[{pid}] 完成 batches={batch_count + 1}, papers={processed}, errors={errors}")
        print(f"[{pid}] 批次文件保存在: {temp_folder}")
    
    except Exception as e:
        print(f"[{pid}] 处理失败: {e}")
        import traceback
        traceback.print_exc()
        
        # 失败时保留临时文件以便排查
        print(f"[{pid}] 临时文件保存在: {temp_folder}")

# =====================================================
# 获取已完成part的最大编号
# =====================================================
def get_max_completed_part(edge_folder):
    """
    从entities_pair文件夹中查找所有已完成的part：
    1. edge_part_XXX.gz 文件
    2. temp_XXX 文件夹
    返回最大的part编号
    """
    max_part = -1
    
    # 1. 查找所有 edge_part_XXX.gz 文件
    edge_files = glob.glob(os.path.join(edge_folder, 'edge_part_*.gz'))
    for edge_file in edge_files:
        part_num = extract_part_id_from_path(edge_file)
        if part_num is not None and part_num > max_part:
            max_part = part_num
            print(f"发现edge文件: {os.path.basename(edge_file)}, part={part_num}")
    
    # 2. 查找所有 temp_XXX 文件夹
    temp_folders = glob.glob(os.path.join(edge_folder, 'temp_*'))
    for temp_folder in temp_folders:
        if os.path.isdir(temp_folder):
            part_num = extract_part_id_from_path(temp_folder)
            if part_num is not None and part_num > max_part:
                max_part = part_num
                print(f"发现temp文件夹: {os.path.basename(temp_folder)}, part={part_num}")
    
    return max_part

# =====================================================
# 辅助函数：检查part是否已完成
# =====================================================
def is_part_completed(part_id, edge_folder):
    """
    检查指定part是否已完成
    """
    pid = f"{part_id:03d}"
    edge_file = os.path.join(edge_folder, f'edge_part_{pid}.gz')
    temp_folder = os.path.join(edge_folder, f'temp_{pid}')
    
    # 如果存在edge文件或temp文件夹，认为已完成
    return os.path.exists(edge_file) or os.path.exists(temp_folder)

# =====================================================
# ENTRY POINT
# =====================================================
if __name__ == "__main__":
    base_folder = "G:\\openalex-snapshot\\data\\works"
    concept_file = "E:\\study\\research\\ASIST\\entities.txt"

    edge_folder = "H:\\ASIST\\entities_pair"
    log_folder = "H:\\ASIST\\logs_pair"
    os.makedirs(edge_folder, exist_ok=True)
    os.makedirs(log_folder, exist_ok=True)

    files = sorted(glob.glob(f"{base_folder}/updated_date=*/part_*.gz"))

    # =====================================================
    # 加载概念 + 构建 AC 自动机
    # =====================================================
    with open(concept_file, 'r', encoding='utf-8') as f:
        full_concepts = [c.strip().lower() for c in f if c.strip()]
    print(f"概念数量: {len(full_concepts)}")

    ac_automaton = ACAutomaton()
    for idx, concept in enumerate(full_concepts):
        ac_automaton.add_word(concept, idx)
    ac_automaton.build()
    print("AC 自动机构建完成")

    # =====================================================
    # 断点续跑：获取已完成part的最大编号
    # =====================================================
    print("\n正在扫描已完成的part...")
    max_completed_part = get_max_completed_part(edge_folder)
    
    if max_completed_part >= 0:
        print(f"\n检测到已完成的最大 part 编号: {max_completed_part:03d}")
        print(f"将从 part {max_completed_part + 1:03d} 开始处理")
        
        # 显示将要跳过的parts
        skipped_parts = []
        for i in range(max_completed_part + 1):
            if is_part_completed(i, edge_folder):
                skipped_parts.append(f"{i:03d}")
        if skipped_parts:
            print(f"将跳过以下已完成的parts: {', '.join(skipped_parts)}")
    else:
        print("\n未检测到已完成文件，将从 part 000 开始处理")
        max_completed_part = -1

    # =====================================================
    # 顺序处理每个文件
    # =====================================================
    total_tasks = 0
    processed_count = 0
    skipped_count = 0
    
    print(f"\n总文件数: {len(files)}")
    
    for i, f in enumerate(files):
        # 如果当前part编号 <= 最大已完成编号，则跳过
        if i <= max_completed_part:
            if is_part_completed(i, edge_folder):
                skipped_count += 1
            continue
        
        total_tasks += 1
        print(f"\n开始处理第 {total_tasks} 个新任务: part {i:03d}")
        process_file(f, i, edge_folder, log_folder, ac_automaton)
        processed_count += 1
        
        # 每处理完一个文件，更新当前进度信息
        print(f"\n进度: 已完成 {processed_count}/{total_tasks} 个新任务")

    print(f"\n{'='*50}")
    print(f"处理完成统计:")
    print(f"  总文件数: {len(files)}")
    print(f"  已跳过: {skipped_count} 个已完成文件")
    print(f"  本次新增处理: {processed_count} 个文件")
    print(f"  当前已完成的最大 part 编号: {max_completed_part + processed_count:03d}")
    print(f"{'='*50}")
    
    with open("job_finish.txt", 'a', encoding='utf-8') as f:
        f.write(f"\n{'='*50}\n")
        f.write(f"All done: {datetime.now()}\n")
        f.write(f"Total files: {len(files)}\n")
        f.write(f"Skipped: {skipped_count}\n")
        f.write(f"Processed: {processed_count}\n")
        f.write(f"Last completed part: {max_completed_part + processed_count:03d}\n")
        f.write(f"{'='*50}\n")

    print("\n=== 全部任务完成 ===")

In [ ]:
import glob
import gzip
import json
import os
import pickle
import re
from datetime import datetime, date
from functools import reduce
from collections import defaultdict
from ahocorasick import Automaton

# =====================================================
# AC AUTOMATON
# =====================================================
class ACAutomaton:
    def __init__(self):
        self.goto = [{}]
        self.out = [set()]
        self.fail = [0]

    def add_word(self, word, index):
        state = 0
        for char in word:
            if char not in self.goto[state]:
                self.goto[state][char] = len(self.goto)
                self.goto.append({})
                self.out.append(set())
                self.fail.append(0)
            state = self.goto[state][char]
        self.out[state].add(index)

    def build(self):
        queue = []
        for char, state in self.goto[0].items():
            queue.append(state)
        while queue:
            r = queue.pop(0)
            for char, s in self.goto[r].items():
                queue.append(s)
                f = self.fail[r]
                while f and char not in self.goto[f]:
                    f = self.fail[f]
                self.fail[s] = self.goto[f].get(char, 0)
                self.out[s] |= self.out[self.fail[s]]

    def search(self, text):
        state = 0
        results = set()
        for char in text:
            while state and char not in self.goto[state]:
                state = self.fail[state]
            state = self.goto[state].get(char, 0)
            if self.out[state]:
                results |= self.out[state]
        return results

# =====================================================
# HELPERS
# =====================================================
def safe_json_loads(line):
    try:
        return json.loads(line)
    except:
        return None

def safe_date_parsing(date_str):
    try:
        return datetime.strptime(date_str, "%Y-%m-%d").date()
    except:
        return None

def get_single_article_string(article):
    try:
        title = article.get('title', '') or ''
        inv = article.get('abstract_inverted_index')
        if not inv:
            abstract = ""
        else:
            pairs = []
            for w, ps in inv.items():
                for p in ps or []:
                    pairs.append((p, w))
            abstract = ' '.join(w for _, w in sorted(pairs))

        replace_pairs = [
            ['\n', ' '], ['-', ' '], ['  ', ' ']
        ]

        text = (title + ' ' + abstract).lower()
        for a, b in replace_pairs:
            text = text.replace(a, b)

        return text
    except:
        return ""

def extract_id(filename):
    m = re.search(r'part_(\d+)', filename)
    return int(m.group(1)) if m else None

def extract_part_id_from_path(path):
    """
    从文件路径或文件夹名称中提取part编号
    支持格式: edge_part_123.gz, temp_123, edge_part_123.pkl.gz 等
    """
    basename = os.path.basename(path)
    # 匹配各种格式的part编号
    patterns = [
        r'edge_part_(\d+)\.gz$',     # edge_part_123.gz
        r'temp_(\d+)$',              # temp_123
        
    ]
    
    for pattern in patterns:
        match = re.search(pattern, basename)
        if match:
            return int(match.group(1))
    return None

# =====================================================
# PROCESS SINGLE FILE
# =====================================================
def process_file(file_path, part_id, edge_folder, log_folder, ac_automaton):
    pid = f"{part_id:03d}"
    edge_file = os.path.join(edge_folder, f'edge_part_{pid}.gz')
    finish_log = os.path.join(log_folder, f'log_edge_part_{pid}_finish.txt')
    temp_folder = os.path.join(edge_folder, f'temp_{pid}')
    
    # Checkpoint logic: resume if this part is completed
    if os.path.exists(finish_log) or os.path.exists(edge_file) or os.path.exists(temp_folder):
        print(f"[{pid}] 已存在（edge文件、完成日志或temp文件夹），跳过")
        return
    
    # 创建临时文件夹
    os.makedirs(temp_folder, exist_ok=True)
    
    processed, errors = 0, 0
    paper_starting_date = date(1990, 1, 1)
    
    # 分批处理参数
    batch_size = 5000
    max_batches = 3000  # 最大批次数量限制
    batch_count = 0
    temp_files = []
    current_batch = []
    
    try:
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                article = safe_json_loads(line)
                if not article:
                    errors += 1
                    continue
                
                pub_date = safe_date_parsing(article.get('publication_date', ''))
                if not pub_date:
                    errors += 1
                    continue
                
                text = get_single_article_string(article)
                if not text:
                    continue
                
                matches = ac_automaton.search(text)
                if len(matches) < 2:
                    continue
                
                curr_time = (pub_date - paper_starting_date).days
                curr_cite = article.get('cited_by_count', 0)
                curr_cite_year = article.get('counts_by_year', [])
                
                mc = sorted(matches)
                for i in range(len(mc)):
                    for j in range(i + 1, len(mc)):
                        current_batch.append([mc[i], mc[j], curr_time, curr_cite, curr_cite_year])
                
                processed += 1
                
                # 批次大小达到阈值，保存到临时文件
                if len(current_batch) >= batch_size:
                    temp_file = os.path.join(temp_folder, f'batch_{batch_count:04d}.pkl.gz')
                    with gzip.open(temp_file, 'wb') as tf:
                        pickle.dump(current_batch, tf)
                    temp_files.append(temp_file)
                    current_batch = []
                    batch_count += 1
                    print(f"[{pid}] 已保存批次 {batch_count}，累计处理文章 {processed}")
                    
                    # 检查是否达到最大批次限制
                    if batch_count >= max_batches:
                        print(f"[{pid}] 已达到最大批次限制 ({max_batches})，停止处理")
                        break

        # 保存最后一批（如果有且未达到批次限制）
        if current_batch and batch_count < max_batches:
            temp_file = os.path.join(temp_folder, f'batch_{batch_count:04d}.pkl.gz')
            with gzip.open(temp_file, 'wb') as tf:
                pickle.dump(current_batch, tf)
            temp_files.append(temp_file)
            batch_count += 1
        
        # 写完成日志
        with open(finish_log, 'w', encoding='utf-8') as f:
            f.write(f"Finish Time: {datetime.now()}\n")
            f.write(f"Total batches: {batch_count}\n")
            f.write(f"Processed papers: {processed}\n")
            f.write(f"Errors: {errors}\n")
            f.write(f"Temp folder: {temp_folder}\n")
            if batch_count >= max_batches:
                f.write(f"Note: Reached maximum batch limit ({max_batches}), processing stopped early\n")
        
        print(f"[{pid}] 完成 batches={batch_count}, papers={processed}, errors={errors}")
        print(f"[{pid}] 批次文件保存在: {temp_folder}")
    
    except Exception as e:
        print(f"[{pid}] 处理失败: {e}")
        import traceback
        traceback.print_exc()
        
        # 失败时保留临时文件以便排查
        print(f"[{pid}] 临时文件保存在: {temp_folder}")

# =====================================================
# 获取已完成part的最大编号
# =====================================================
def get_max_completed_part(edge_folder):
    """
    从entities_pair文件夹中查找所有已完成的part：
    1. edge_part_XXX.gz 文件
    2. temp_XXX 文件夹
    返回最大的part编号
    """
    max_part = -1
    
    # 1. 查找所有 edge_part_XXX.gz 文件
    edge_files = glob.glob(os.path.join(edge_folder, 'edge_part_*.gz'))
    for edge_file in edge_files:
        part_num = extract_part_id_from_path(edge_file)
        if part_num is not None and part_num > max_part:
            max_part = part_num
            print(f"发现edge文件: {os.path.basename(edge_file)}, part={part_num}")
    
    # 2. 查找所有 temp_XXX 文件夹
    temp_folders = glob.glob(os.path.join(edge_folder, 'temp_*'))
    for temp_folder in temp_folders:
        if os.path.isdir(temp_folder):
            part_num = extract_part_id_from_path(temp_folder)
            if part_num is not None and part_num > max_part:
                max_part = part_num
                print(f"发现temp文件夹: {os.path.basename(temp_folder)}, part={part_num}")
    
    return max_part

# =====================================================
# 辅助函数：检查part是否已完成
# =====================================================
def is_part_completed(part_id, edge_folder):
    """
    检查指定part是否已完成
    """
    pid = f"{part_id:03d}"
    edge_file = os.path.join(edge_folder, f'edge_part_{pid}.gz')
    temp_folder = os.path.join(edge_folder, f'temp_{pid}')
    
    # 如果存在edge文件或temp文件夹，认为已完成
    return os.path.exists(edge_file) or os.path.exists(temp_folder)

# =====================================================
# ENTRY POINT
# =====================================================
if __name__ == "__main__":
    base_folder = "G:\\openalex-snapshot\\data\\works"
    concept_file = "E:\\study\\research\\ASIST\\entities.txt"

    edge_folder = "H:\\ASIST\\entities_pair"
    log_folder = "H:\\ASIST\\logs_pair"
    os.makedirs(edge_folder, exist_ok=True)
    os.makedirs(log_folder, exist_ok=True)

    files = sorted(glob.glob(f"{base_folder}/updated_date=*/part_*.gz"))

    # =====================================================
    # 加载概念 + 构建 AC 自动机
    # =====================================================
    with open(concept_file, 'r', encoding='utf-8') as f:
        full_concepts = [c.strip().lower() for c in f if c.strip()]
    print(f"概念数量: {len(full_concepts)}")

    ac_automaton = ACAutomaton()
    for idx, concept in enumerate(full_concepts):
        ac_automaton.add_word(concept, idx)
    ac_automaton.build()
    print("AC 自动机构建完成")

    # =====================================================
    # 断点续跑：获取已完成part的最大编号
    # =====================================================
    print("\n正在扫描已完成的part...")
    max_completed_part = get_max_completed_part(edge_folder)
    
    if max_completed_part >= 0:
        print(f"\n检测到已完成的最大 part 编号: {max_completed_part:03d}")
        print(f"将从 part {max_completed_part + 1:03d} 开始处理")
        
        # 显示将要跳过的parts
        skipped_parts = []
        for i in range(max_completed_part + 1):
            if is_part_completed(i, edge_folder):
                skipped_parts.append(f"{i:03d}")
        if skipped_parts:
            print(f"将跳过以下已完成的parts: {', '.join(skipped_parts)}")
    else:
        print("\n未检测到已完成文件，将从 part 000 开始处理")
        max_completed_part = -1

    # =====================================================
    # 顺序处理每个文件
    # =====================================================
    total_tasks = 0
    processed_count = 0
    skipped_count = 0
    
    print(f"\n总文件数: {len(files)}")
    
    for i, f in enumerate(files):
        # 如果当前part编号 <= 最大已完成编号，则跳过
        if i <= max_completed_part:
            if is_part_completed(i, edge_folder):
                skipped_count += 1
            continue
        
        total_tasks += 1
        print(f"\n开始处理第 {total_tasks} 个新任务: part {i:03d}")
        process_file(f, i, edge_folder, log_folder, ac_automaton)
        processed_count += 1
        
        # 每处理完一个文件，更新当前进度信息
        print(f"\n进度: 已完成 {processed_count}/{total_tasks} 个新任务")

    print(f"\n{'='*50}")
    print(f"处理完成统计:")
    print(f"  总文件数: {len(files)}")
    print(f"  已跳过: {skipped_count} 个已完成文件")
    print(f"  本次新增处理: {processed_count} 个文件")
    print(f"  当前已完成的最大 part 编号: {max_completed_part + processed_count:03d}")
    print(f"{'='*50}")
    
    with open("job_finish.txt", 'a', encoding='utf-8') as f:
        f.write(f"\n{'='*50}\n")
        f.write(f"All done: {datetime.now()}\n")
        f.write(f"Total files: {len(files)}\n")
        f.write(f"Skipped: {skipped_count}\n")
        f.write(f"Processed: {processed_count}\n")
        f.write(f"Last completed part: {max_completed_part + processed_count:03d}\n")
        f.write(f"{'='*50}\n")

    print("\n=== 全部任务完成 ===")

In [ ]:
! pip install pyahocorasick

In [ ]:
# 推荐在 conda / venv 下
! pip install -U sentence-transformers faiss-gpu torch tqdm
# 如果 faiss-gpu 安装出问题，根据你的cuda版本选择合适的faiss包（比如 conda install -c pytorch faiss-gpu）
! pip install ahocorasick

对概念共现边数据进行质量过滤，从所有概念关联边中筛选出只包含有实际引用（引用数不为零）的论文中的概念对。它会读取原始边数据文件，移除无人引用的论文产生的概念关联，将过滤后的高质量边数据保存到新的文件夹中，并统计显示过滤前后的数据量变化，确保后续分析基于有学术影响力的概念关系。

In [ ]:
import os
import gzip
import pickle

# === 设置你的文件夹路径 ===
folder_path = "H:\\concept_pair"

# 输出路径（保存筛选后的结果）
output_folder = os.path.join(folder_path, "filtered")
os.makedirs(output_folder, exist_ok=True)

# 获取所有 .gz 文件
gz_files = [f for f in os.listdir(folder_path) if f.endswith(".gz")]
print(f"共找到 {len(gz_files)} 个文件。")

total_before = 0
total_after = 0

# 循环处理每个文件
for gz_file in gz_files:
    file_path = os.path.join(folder_path, gz_file)
    output_path = os.path.join(output_folder, gz_file)

    try:
        with gzip.open(file_path, "rb") as f:
            edge_list = pickle.load(f)
    except Exception as e:
        print(f" 无法读取 {gz_file}: {e}")
        continue

    total_before += len(edge_list)

    # 筛选：共被引次数不为 0
    filtered = [
        edge for edge in edge_list
        if len(edge) > 3 and isinstance(edge[3], (int, float)) and edge[3] != 0
    ]

    total_after += len(filtered)

    # 打印前几条看看
    print(f"\n {gz_file}")
    print(f"原始记录: {len(edge_list)}  →  筛选后: {len(filtered)}")
    if filtered:
        for e in filtered[:5]:
            print(e)

        # 保存筛选结果
        with gzip.open(output_path, "wb") as f:
            pickle.dump(filtered, f)
    else:
        print("无符合条件的记录，跳过保存。")

# print("\n=== 全部完成 ===")
# print(f"总记录数: {total_before} → 保留 {total_after} （{total_after/total_before*100:.2f}%）")
# print(f"筛选后文件保存在: {output_folder}")